# Phase 2B — Reliable GPU Accuracy–Runtime Comparison

Reproducible interleaved benchmark: **six locally repeated sessions**, **30 randomized-order blocks per session**, and **four input sizes**. Online algorithms are unchanged from Phase 1. The `RUN_FORMAL` switch is **OFF by default**; the CPU smoke and offline report do not need a GPU.

The notebook loads code from the public GitHub `revision/major-experiments` branch and records the exact commit. Do not compare timings across unlike GPUs. No private reviewer correspondence is included.

### A · Setup — clone or locate repository, install editable package

In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path.cwd().resolve()
if not (repo / "revision/phase2/benchmarks/run_benchmark_reliability.py").exists():
    repo = Path("/content/Fast_Lp_Prox")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "revision/major-experiments",
                        "https://github.com/Jy-U1/Fast_Lp_Prox.git", str(repo)], check=True)
assert (repo / "revision/phase2/benchmarks/run_benchmark_reliability.py").exists(), "Phase 2B not found: pull/push your branch first"
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], check=True)
sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repo:", repo, "\nCommit:", sha)
try:
    import torch
    print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch diagnostics unavailable:", exc)

### B · CPU smoke (optional; not a GPU result)

Run once to verify the new interleaving, accuracy metrics, summary and charts.

In [ ]:
from datetime import datetime, timezone
RUN_SMOKE = True
SMOKE = repo / "results/phase2" / ("reliability_smoke_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SZ"))
if RUN_SMOKE:
    subprocess.run([sys.executable, "revision/phase2/benchmarks/run_benchmark_reliability.py",
        "--device", "cpu", "--p-grid", "1/3", "--sizes", "512,2048",
        "--sessions", "2", "--blocks", "4", "--warmup", "1",
        "--fit-n", "256", "--validation-n", "501", "--accuracy-n", "512",
        "--output-dir", str(SMOKE)], check=True)
    subprocess.run([sys.executable, "revision/phase2/benchmarks/rebuild_benchmark_artifacts.py",
        "--root", str(SMOKE), "--bootstrap-resamples", "100"], check=True)
    print("CPU smoke folder:", SMOKE)
else:
    print("CPU smoke skipped")

### C · Formal GPU benchmark (opt-in)

Before enabling, select an NVIDIA GPU in **Runtime → Change runtime type**. If you want to compare to Phase 1, use the **same GPU model** where possible. This code keeps each individual timing sample and its within-block order. Execution may take several minutes depending on host and GPU.

In [ ]:
RUN_FORMAL = False   # change to True for the formal GPU session
FORMAL = repo / "results/phase2/benchmark_reliability" / (
    "formal_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SZ"))
if RUN_FORMAL:
    import torch
    assert torch.cuda.is_available(), "Enable a CUDA GPU in Colab first"
    subprocess.run([sys.executable, "revision/phase2/benchmarks/run_benchmark_reliability.py",
        "--device", "cuda", "--dtype", "float32",
        "--p-grid", "1/3,2/5,3/4", "--sizes", "10000,100000,1000000,10000000",
        "--sessions", "6", "--blocks", "30", "--warmup", "10",
        "--accuracy-n", "8192", "--target-norm-error", "1e-6",
        "--fit-n", "8192", "--validation-n", "80001",
        "--output-dir", str(FORMAL)], check=True)
    subprocess.run([sys.executable, "revision/phase2/benchmarks/rebuild_benchmark_artifacts.py",
                    "--root", str(FORMAL)], check=True)
    print("Formal GPU folder:", FORMAL)
else:
    print("Formal GPU run skipped. Change RUN_FORMAL to True when ready.")

### D · Inspect results, accuracy gate, per-session variation

In [ ]:
import pandas as pd
from IPython.display import display, Image
ROOT = FORMAL if (RUN_FORMAL and (FORMAL / "paired_summary.csv").exists()) else SMOKE
assert (ROOT / "paired_summary.csv").exists(), "Run Section B or C first"
summary = pd.read_csv(ROOT / "paired_summary.csv")
quality = pd.read_csv(ROOT / "accuracy_checks.csv")
print("Data root:", ROOT)
print("Failed accuracy subset checks:", int((~quality["accuracy_pass"]).sum()))
cols = ["p_label", "n", "cheb_config", "newton_config",
        "median_cheb_ms", "median_newton_ms", "median_paired_ratio_cheb_over_newton",
        "ci95_low_ratio", "ci95_high_ratio", "all_accuracy_checks_pass"]
display(summary[cols])
display(Image(filename=str(ROOT / "figures/paired_ratio_vs_size.png")))
if not quality["accuracy_pass"].all():
    print("WARNING: at least one candidate/size/session did not pass the output-accuracy gate; do not call those groups accuracy-matched.")

### E · Export ZIP (preserves full raw individual timings + source code)

The archive is a research record, **not** something to commit to Git. Store it in your local `Revision/Archives/` after validation.

In [ ]:
import zipfile, json, hashlib
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SZ")
archive = repo / "results/phase2" / f"FastLp_Phase2B_Reliability_Export_{stamp}.zip"
archive.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=7) as z:
    for path in sorted(ROOT.rglob("*")):
        if path.is_file():
            z.write(path, "phase2b/results/" + path.relative_to(ROOT).as_posix())
    for path in sorted((repo / "revision/phase2/benchmarks").glob("*.py")):
        z.write(path, "phase2b/source/" + path.name)
    z.write(repo / "revision/phase2/benchmarks/README.md", "phase2b/source/README.md")
    z.write(repo / "revision/phase2/notebooks/Phase2B_Benchmark_Reliability.ipynb",
            "phase2b/source/Phase2B_Benchmark_Reliability.ipynb")
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    print("Export complete:", archive, "entries:", len(z.namelist()))
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("ZIP is ready on the local filesystem")